# 📊 Pocket OTC AI Analyzer — Google Colab

## تشغيل بنقرة واحدة
هذه النسخة تحتوي على خلية تشغيل واحدة فقط: تنزيل المشروع، تثبيت المتطلبات، طلب المفاتيح، تشغيل Web UI وTelegram Bot، ثم اختبار النظام.

**READ-ONLY:** لا يسجل الدخول إلى Pocket Option ولا ينفذ أوامر تداول.

In [ ]:
# 🚀 تشغيل كامل بنقرة واحدة
import os, sys, shutil, subprocess, time, socket, threading, traceback, importlib
from getpass import getpass

ROOT = '/content/Jjjjjjj'
REPO = 'https://github.com/mohmb142/Jjjjjjj.git'
PORT = 8000

try:
    if os.path.exists(ROOT):
        shutil.rmtree(ROOT, ignore_errors=True)

    print('1/6 ⬇️ تنزيل آخر نسخة من GitHub...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO, ROOT], check=True)
    if not os.path.isfile(os.path.join(ROOT, 'main.py')):
        raise RuntimeError('main.py غير موجود في المستودع')
    os.chdir(ROOT)
    if ROOT not in sys.path:
        sys.path.insert(0, ROOT)
    for module_name in ('main', 'telegram_bot', 'image_analyzer', 'openrouter', 'pocket_data', 'signal_engine', 'indicators'):
        sys.modules.pop(module_name, None)

    print('2/6 📦 تثبيت المتطلبات وفحص الملفات...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'colab_requirements.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'compileall', '-q', '.'], check=True)

    print('3/6 🔐 أدخل المفاتيح عند الطلب...')
    telegram_token = getpass('Telegram Bot Token: ').strip()
    openrouter_key = getpass('OpenRouter API Key: ').strip()
    if not telegram_token or not openrouter_key:
        raise ValueError('يجب إدخال Telegram Bot Token و OpenRouter API Key')
    os.environ['TELEGRAM_BOT_TOKEN'] = telegram_token
    os.environ['OPENROUTER_API_KEY'] = openrouter_key
    os.environ['OPENROUTER_MODEL'] = 'google/gemini-2.5-flash'

    print('4/6 🌐 تشغيل FastAPI...')
    import uvicorn
    import main

    def web_worker():
        try:
            uvicorn.run(main.app, host='0.0.0.0', port=PORT, log_level='info')
        except Exception:
            traceback.print_exc()

    web_thread = threading.Thread(target=web_worker, name='fastapi', daemon=True)
    web_thread.start()

    deadline = time.time() + 45
    while time.time() < deadline:
        try:
            with socket.create_connection(('127.0.0.1', PORT), timeout=0.5):
                break
        except OSError:
            time.sleep(0.25)
    else:
        raise RuntimeError('FastAPI لم تبدأ على المنفذ 8000')

    from google.colab.output import eval_js
    from IPython.display import HTML, display
    public_url = eval_js(f'google.colab.kernel.proxyPort({PORT})')

    print('5/6 🤖 تشغيل Telegram Bot...')
    from telegram_bot import run

    def telegram_worker():
        try:
            run()
        except Exception:
            traceback.print_exc()

    telegram_thread = threading.Thread(target=telegram_worker, name='telegram-bot', daemon=True)
    telegram_thread.start()
    time.sleep(2)

    print('6/6 🧪 اختبار النظام...')
    import requests
    response = requests.get('http://127.0.0.1:8000/health', timeout=15)
    response.raise_for_status()

    display(HTML(f'''
    <div style="font-family:Arial;padding:20px;border:2px solid #2e7d32;border-radius:12px">
        <h2>✅ Pocket OTC AI Analyzer يعمل</h2>
        <p>🌐 Web UI</p>
        <a href="{public_url}" target="_blank" style="font-size:20px">🚀 فتح الواجهة</a>
        <p>🤖 Telegram Bot: يعمل</p>
        <p>⏱️ أفق التحليل: دقيقة واحدة</p>
        <p>🔒 الوضع: READ-ONLY</p>
    </div>
    '''))

    print('Health:', response.json())
    print('🌐 الرابط:', public_url)
    print('⚠️ أبقِ جلسة Colab مفتوحة حتى يبقى النظام يعمل.')

except Exception:
    print('❌ فشل التشغيل:')
    traceback.print_exc()
    print('إذا فشل clone أو pip، أعد تشغيل Runtime ثم شغّل الخلية مرة أخرى.')